# Transient Degrees of Freedom

First we import some packages

In [1]:
import numpy as np
import pyPolygon as pp
from matplotlib import pyplot as plt

Then we make our packing. We want it to be a packing of 5 squares. We also make sure this demo is repeatable, so we set a random seed. We also make sure our box is slightly smaller than the theoretical smallest possible (side length = 2.707...)

The packing fraction is therefore 5 / 2.707...^2. We can make the box a little smaller and use 2.7 instead. This will help ensure we're overjammed.

In [ ]:
packing = pp.Model(N = 5, n = 4, seed = 42)
# The default if a seed isn't specified is to generate a random seed
packing.generateEquilateralPolygons(phi = 5 / (2.5)**2, kappa = 4.0)
# We can synchronize the target perimeters and areas:
packing.syncTargetPerimeters()
# Notice that phi is going to be too large so we'll get 
# overlapping squares no matter what.
packing.setLogNormalTargetPerimeter(polydispersity = 0.25) # make all the squares have 
# a target perimeter drawn from a log-normal distribution with std/mean of 0.1

# We then set our fixed boundary conditions
vertices = np.array([[0, 0], [0, 1], [1, 1], [1, 0]])
packing.addShape(vertices)
packing.pinVertices(np.arange(packing.getNumVertices())[-4:])
packing.setBoundaryConditions("fixed")
# We add springs to our areas:
packing.setSpringConstants(area = 0, edge = 1, perimeter = 0)
# We add some constraints:
packing.setConstraints(area = True, edge = False)
# We mollify
packing.setModelType("mollified")
packing.setMollification(sigma = 1e-2)
# And minimize
packing.minimizeFIRE(maxUnbalancedForce = 1e-3, progressBar = True)
packing.draw()

/tmp/ipykernel_66014/3120112719.py:3: UserWarning: phi is large: a polygon spans ~0.57 of the unit box (> 0.5), so the single-image periodic assumption in the overlap machinery may be violated; reduce phi.
  packing.generateEquilateralPolygons(phi = 5 / (2.5)**2, kappa = 4.0)
/home/rdennis/Documents/Code/pyPolygon/model.py:1058: UserWarning: 
*** pyPolygon is falling back to the PYTHON (numpy) tier -- the library loaded but no usable GPU responded ***
    The GPU overlap is 150-650x faster; expect minimization to be very slow.
    A driver/library version mismatch after a driver update is the usual cause -- compare 'cat /proc/driver/nvidia/version' with the installed libcuda; a reboot normally clears it.
  onGpu = cudaOverlap is not None and cudaOverlap.isAvailable()


FIRE (constrained):   0%|          | 0/100000 [00:00<?, ?it/s]

Next we let the areas and edgeLengths be degrees of freedom with a set of fixed moments of the distribution

In [ ]:
packing.setDOFType("transient")
packing.setMoments([1, 2, -1, 4])

In [ ]:
# Now we minimize:
packing.minimizeFIRE(maxUnbalancedForce = 1e-7, progressBar = True)

In [ ]:
packing.draw(forces = packing.getForces())

## Next we create a schedule for reducing phi, polydispersity, and sigma

In [ ]:
packing.energySweep(finalPolydispersity = 1e-5, finalEnergy = 1e-5, progressBar = True)

In [ ]:
packing.draw()